In [1]:
"""
ICPAC Flood and Drought Data Extractor

A specialized tool for extracting flood and drought data from the Montandon STAC API
for ICPAC member countries and organizing it into CSV format.

Requirements:
    pip install pystac-client requests pandas tqdm
"""

import os
import csv
import logging
import requests
import json
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any

import pystac_client
import pandas as pd
from tqdm import tqdm

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ICPAC countries with ISO codes
ICPAC_COUNTRIES = {
    'Djibouti': 'DJI', 'Eritrea': 'ERI', 'Ethiopia': 'ETH', 'Kenya': 'KEN',
    'Somalia': 'SOM', 'South Sudan': 'SSD', 'Sudan': 'SDN', 'Uganda': 'UGA',
    'Burundi': 'BDI', 'Rwanda': 'RWA', 'Tanzania': 'TZA'
}

# STAC API endpoint
MONTANDON_STAC_URL = "https://montandon-eoapi-stage.ifrc.org/stac"

# Country bounding boxes
COUNTRY_BBOXES = {
    'Djibouti': (41.66, 10.9, 43.42, 12.71),
    'Eritrea': (36.32, 12.36, 43.13, 18.03),
    'Ethiopia': (33.99, 3.40, 47.98, 14.89),
    'Kenya': (33.91, -4.67, 41.91, 5.51),
    'Somalia': (40.98, -1.65, 51.13, 11.99),
    'South Sudan': (23.89, 3.49, 35.79, 12.24),
    'Sudan': (21.83, 9.34, 38.61, 22.23),
    'Uganda': (29.57, -1.48, 35.00, 4.23),
    'Burundi': (28.99, -4.45, 30.84, -2.31),
    'Rwanda': (28.85, -2.83, 30.90, -1.05),
    'Tanzania': (29.33, -11.75, 40.45, -0.99),
}

# Keywords to identify flood and drought collections
FLOOD_KEYWORDS = ['flood', 'inundation', 'water extent', 'surface water', 'water detection', 'water body']
DROUGHT_KEYWORDS = [
    'drought', 'dry', 'moisture', 'precipitation', 'rainfall', 'vegetation health',
    'ndvi', 'evi', 'vhi', 'vci', 'tci', 'spi', 'spei', 'soil moisture', 'evapotranspiration',
    'water deficit', 'aridity', 'vegetation stress', 'vegetation condition', 'water scarcity',
    'water stress', 'dryness', 'arid'
]

# Specific collection IDs known to contain drought or flood data
KNOWN_DROUGHT_COLLECTIONS = [
    'modis-ndvi', 'sentinel-ndvi', 'sentinel-evi', 'modis-lst', 
    'smap-soil-moisture', 'chirps-rainfall', 'vhi', 'vci', 'tci',
    'sentinel-s2-l2a', 'landsat-8-l2', 'terra-modis'
]

KNOWN_FLOOD_COLLECTIONS = [
    'water-extent', 'flood-map', 'gfms', 'jrc-gsw', 'surface-water'
]

# Add EM-DAT collection
EMDAT_COLLECTION = 'emdat'

class STACClient:
    """Specialized client for accessing flood and drought data from STAC API."""

    def __init__(self, stac_url=MONTANDON_STAC_URL):
        """Initialize the STAC client."""
        self.stac_url = stac_url
        try:
            self.client = pystac_client.Client.open(stac_url)
            logger.info(f"Connected to STAC API at {stac_url}")
        except Exception as e:
            logger.error(f"Failed to connect to STAC API: {e}")
            raise

    def list_collections(self) -> List[Any]:
        """List all available collections in the STAC catalog."""
        try:
            collections = list(self.client.get_collections())
            logger.info(f"Found {len(collections)} collections")
            return collections
        except Exception as e:
            logger.error(f"Failed to list collections: {e}")
            raise

    def filter_collections_by_hazard(self) -> Dict[str, List[Any]]:
        """Filter collections by hazard type (flood or drought)."""
        collections = self.list_collections()
        filtered = {
            'flood': [],
            'drought': [],
            'emdat': []
        }
        
        for collection in collections:
            collection_info = {
                'id': collection.id,
                'title': collection.title or '',
                'description': collection.description or ''
            }
            
            # Check if collection ID is emdat
            if EMDAT_COLLECTION in collection.id.lower():
                filtered['emdat'].append(collection)
                logger.info(f"Found EM-DAT collection: {collection.id}")
                continue
                
            # Check if collection ID matches known collections
            if collection.id in KNOWN_DROUGHT_COLLECTIONS:
                filtered['drought'].append(collection)
                continue
                
            if collection.id in KNOWN_FLOOD_COLLECTIONS:
                filtered['flood'].append(collection)
                continue
            
            # Check if collection is related to floods
            flood_match = False
            for keyword in FLOOD_KEYWORDS:
                if (keyword.lower() in collection.id.lower() or
                    (collection.title and keyword.lower() in collection.title.lower()) or
                    (collection.description and keyword.lower() in collection.description.lower())):
                    filtered['flood'].append(collection)
                    flood_match = True
                    break
            
            # Skip to next collection if already matched with flood
            if flood_match:
                continue
                    
            # Check if collection is related to droughts
            for keyword in DROUGHT_KEYWORDS:
                if (keyword.lower() in collection.id.lower() or
                    (collection.title and keyword.lower() in collection.title.lower()) or
                    (collection.description and keyword.lower() in collection.description.lower())):
                    filtered['drought'].append(collection)
                    break
        
        # If a collection has both 'sentinel' and 'ndvi' in its id but wasn't caught by keywords, add it to drought
        for collection in collections:
            if (collection not in filtered['drought'] and 
                collection not in filtered['flood'] and 
                collection not in filtered['emdat']):
                if ('ndvi' in collection.id.lower() or 
                    'vegetation' in collection.id.lower() or
                    'rainfall' in collection.id.lower() or
                    'precipitation' in collection.id.lower()):
                    filtered['drought'].append(collection)
        
        logger.info(f"Found {len(filtered['flood'])} flood-related collections")
        logger.info(f"Found {len(filtered['drought'])} drought-related collections")
        logger.info(f"Found {len(filtered['emdat'])} EM-DAT collections")
        
        # Log the collection IDs for debugging
        logger.info(f"Flood collections: {[c.id for c in filtered['flood']]}")
        logger.info(f"Drought collections: {[c.id for c in filtered['drought']]}")
        logger.info(f"EM-DAT collections: {[c.id for c in filtered['emdat']]}")
        
        return filtered

    def print_hazard_collections(self):
        """Print information about available flood, drought and EM-DAT collections."""
        filtered = self.filter_collections_by_hazard()
        
        print("\nFlood-Related Collections:")
        print("=========================")
        for collection in filtered['flood']:
            print(f"ID: {collection.id}")
            print(f"Title: {collection.title or 'No title'}")
            if collection.description:
                desc = collection.description[:100] + "..." if len(collection.description) > 100 else collection.description
                print(f"Description: {desc}")
            print("-" * 50)
            
        print("\nDrought-Related Collections:")
        print("===========================")
        for collection in filtered['drought']:
            print(f"ID: {collection.id}")
            print(f"Title: {collection.title or 'No title'}")
            if collection.description:
                desc = collection.description[:100] + "..." if len(collection.description) > 100 else collection.description
                print(f"Description: {desc}")
            print("-" * 50)
            
        print("\nEM-DAT Collections:")
        print("=================")
        for collection in filtered['emdat']:
            print(f"ID: {collection.id}")
            print(f"Title: {collection.title or 'No title'}")
            if collection.description:
                desc = collection.description[:100] + "..." if len(collection.description) > 100 else collection.description
                print(f"Description: {desc}")
            print("-" * 50)

    def get_collection(self, collection_id: str) -> Any:
        """Get a specific collection by ID."""
        try:
            return self.client.get_collection(collection_id)
        except Exception as e:
            logger.error(f"Failed to get collection {collection_id}: {e}")
            raise

    def search_items(self, collection_id: str, country_name: str,
                    date_range: Optional[Tuple[str, str]] = None) -> List[Any]:
        """Search for items in a collection for a specific country."""
        try:
            bbox = COUNTRY_BBOXES.get(country_name)
            if not bbox:
                raise ValueError(f"Bounding box for {country_name} not available")

            search_params = {
                "collections": [collection_id],
                "bbox": bbox,
            }

            if date_range:
                start_date, end_date = date_range
                search_params["datetime"] = f"{start_date}/{end_date}"

            search = self.client.search(**search_params)
            items = list(search.get_items())

            logger.info(f"Found {len(items)} items for {country_name} in collection {collection_id}")
            return items

        except Exception as e:
            logger.error(f"Failed to search items for {country_name}: {e}")
            raise

    def search_all_countries(self, collection_id: str,
                           date_range: Optional[Tuple[str, str]] = None) -> Dict[str, List[Any]]:
        """Search for items in a collection for all ICPAC countries."""
        results = {}
        total_items = 0

        for country in tqdm(ICPAC_COUNTRIES, desc=f"Searching {collection_id}"):
            try:
                items = self.search_items(collection_id, country, date_range)
                results[country] = items
                total_items += len(items)
            except Exception as e:
                logger.error(f"Error searching items for {country}: {e}")
                results[country] = []
        
        logger.info(f"Found total of {total_items} items across all countries for collection {collection_id}")
        return results


class CSVDataProcessor:
    """Process STAC data and export to CSV format."""

    def __init__(self, client: STACClient):
        """Initialize the data processor."""
        self.client = client

    def extract_item_data(self, item: Any, country: str, hazard_type: str) -> Dict[str, Any]:
        """Extract relevant data from a STAC item."""
        # Base data with default values
        data = {
            'country': country,
            'country_code': ICPAC_COUNTRIES.get(country, ''),
            'hazard_type': hazard_type,
            'collection_id': item.collection_id if hasattr(item, 'collection_id') else '',
            'item_id': item.id if hasattr(item, 'id') else '',
            'datetime': '',
            'date': '',
            'year': '',
            'month': '',
            'bbox': '',
            'assets': '',
        }
        
        # Extract datetime information
        if hasattr(item, 'datetime') and item.datetime:
            data['datetime'] = item.datetime.isoformat()
            data['date'] = item.datetime.strftime('%Y-%m-%d')
            data['year'] = item.datetime.year
            data['month'] = item.datetime.month
        # Try to extract from properties if datetime attribute is not available
        elif hasattr(item, 'properties') and 'datetime' in item.properties:
            try:
                dt = datetime.fromisoformat(item.properties['datetime'].replace('Z', '+00:00'))
                data['datetime'] = dt.isoformat()
                data['date'] = dt.strftime('%Y-%m-%d')
                data['year'] = dt.year
                data['month'] = dt.month
            except (ValueError, AttributeError):
                pass
        
        # Extract bounding box
        if hasattr(item, 'bbox') and item.bbox:
            data['bbox'] = ','.join(map(str, item.bbox))
        
        # Extract assets
        if hasattr(item, 'assets'):
            data['assets'] = ','.join(item.assets.keys())
        
        # Extract properties
        if hasattr(item, 'properties') and item.properties:
            for key, value in item.properties.items():
                # Skip complex objects and create simplified string representations
                if isinstance(value, (dict, list)):
                    continue
                # Clean the key by replacing special characters
                clean_key = key.replace(':', '_').replace('.', '_').lower()
                data[f'prop_{clean_key}'] = str(value)
                
        # For drought items, extract specific drought indicators if available
        if hazard_type == 'drought':
            self._extract_drought_indicators(item, data)
                
        # For EM-DAT items, extract specific disaster information
        if hazard_type == 'emdat':
            self._extract_emdat_data(item, data)
                
        return data
        
    def _extract_drought_indicators(self, item: Any, data: Dict[str, Any]):
        """Extract specific drought indicators from item properties."""
        if not hasattr(item, 'properties'):
            return
            
        props = item.properties
        
        # Look for common drought indicators
        drought_indicators = {
            'ndvi': ['ndvi', 'normalized_difference_vegetation_index'],
            'precipitation': ['precipitation', 'rainfall', 'precip'],
            'soil_moisture': ['soil_moisture', 'soilmoisture', 'sm'],
            'evi': ['evi', 'enhanced_vegetation_index'],
            'vhi': ['vhi', 'vegetation_health_index'],
            'vci': ['vci', 'vegetation_condition_index'],
            'tci': ['tci', 'temperature_condition_index'],
            'spi': ['spi', 'standardized_precipitation_index'],
        }
        
        # Try to extract each indicator
        for indicator_name, possible_keys in drought_indicators.items():
            for key in possible_keys:
                # Look for exact match
                if key in props:
                    data[f'drought_{indicator_name}'] = props[key]
                    break
                # Look for partial match
                for prop_key in props:
                    if key in prop_key.lower():
                        data[f'drought_{indicator_name}'] = props[prop_key]
                        break
    
    def _extract_emdat_data(self, item: Any, data: Dict[str, Any]):
        """Extract specific EM-DAT data from item properties."""
        if not hasattr(item, 'properties'):
            return
            
        props = item.properties
        
        # Common EM-DAT fields
        emdat_fields = [
            'disaster_type', 'disaster_subtype', 'event_name', 'start_date', 'end_date',
            'total_deaths', 'total_affected', 'total_damages', 'disaster_no'
        ]
        
        # Extract EM-DAT specific fields
        for field in emdat_fields:
            for prop_key in props:
                if field.lower() in prop_key.lower():
                    data[f'emdat_{field}'] = props[prop_key]
                    break

    def save_items_to_csv(self, items_dict: Dict[str, List[Any]], 
                          hazard_type: str, 
                          output_file: str):
        """Save items to a CSV file."""
        try:
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            
            all_data = []
            
            # Extract data from all items
            for country, items in items_dict.items():
                for item in items:
                    item_data = self.extract_item_data(item, country, hazard_type)
                    # Only include items with valid dates within our range (2022-01-01 onward)
                    if item_data.get('datetime'):
                        # Check if the date is >= 2022-01-01
                        try:
                            item_date = item_data.get('date', '')
                            if item_date and datetime.strptime(item_date, '%Y-%m-%d') >= datetime.strptime('2022-01-01', '%Y-%m-%d'):
                                all_data.append(item_data)
                        except (ValueError, TypeError):
                            # If date parsing fails, include the item anyway
                            all_data.append(item_data)
                    else:
                        # Include items without datetime (common in EM-DAT)
                        all_data.append(item_data)
            
            if not all_data:
                logger.warning(f"No valid data found for {hazard_type}")
                return
            
            # Get all possible columns
            all_columns = set()
            for data in all_data:
                all_columns.update(data.keys())
            
            # Create a DataFrame
            df = pd.DataFrame(all_data)
            
            # Sort columns for better readability
            essential_columns = ['country', 'country_code', 'hazard_type', 'collection_id', 
                               'item_id', 'datetime', 'date', 'year', 'month', 'bbox', 'assets']
            property_columns = sorted([col for col in all_columns if col.startswith('prop_')])
            emdat_columns = sorted([col for col in all_columns if col.startswith('emdat_')])
            drought_columns = sorted([col for col in all_columns if col.startswith('drought_')])
            
            columns_order = essential_columns + emdat_columns + drought_columns + property_columns
            
            # Reorder columns (only include those that exist in the DataFrame)
            existing_columns = [col for col in columns_order if col in df.columns]
            df = df[existing_columns]
            
            # Save to CSV
            df.to_csv(output_file, index=False)
            logger.info(f"Saved {len(all_data)} items to {output_file}")
            
            return df
            
        except Exception as e:
            logger.error(f"Failed to save items to CSV: {e}")
            raise


class HazardDataExtractor:
    """Extract flood and drought data for ICPAC countries."""

    def __init__(self, client: STACClient, processor: CSVDataProcessor):
        """Initialize the extractor."""
        self.client = client
        self.processor = processor

    def extract_data_for_collection(self, collection_id: str, hazard_type: str,
                                   date_range: Optional[Tuple[str, str]] = None,
                                   output_dir: str = "data"):
        """Extract data for a collection and save to CSV."""
        try:
            # Log extraction information
            start_date, end_date = date_range if date_range else ("", "")
            logger.info(f"Extracting {hazard_type} data from collection '{collection_id}' ({start_date} to {end_date})")
            
            items_dict = self.client.search_all_countries(collection_id, date_range)
            
            # Create a filename based on collection and hazard type
            os.makedirs(output_dir, exist_ok=True)
            output_file = os.path.join(output_dir, f"{hazard_type}_{collection_id}.csv")
            
            # Save to CSV
            df = self.processor.save_items_to_csv(items_dict, hazard_type, output_file)
            
            return output_file, len(df) if df is not None else 0
            
        except Exception as e:
            logger.error(f"Failed to extract data for collection {collection_id}: {e}")
            return None, 0

    def extract_all_hazard_data(self, date_range: Optional[Tuple[str, str]] = None,
                              output_dir: str = "hazard_data"):
        """Extract data for all flood and drought collections."""
        try:
            # Create output directory
            os.makedirs(output_dir, exist_ok=True)
            
            # Get flood and drought collections
            filtered_collections = self.client.filter_collections_by_hazard()
            
            # Set default date range if not provided
            if date_range is None:
                end_date = datetime.now().strftime("%Y-%m-%d")
                start_date = "2022-01-01"  # From January 1, 2022
                date_range = (start_date, end_date)
            
            # Create summary file
            summary_file = os.path.join(output_dir, "extraction_summary.csv")
            
            summary_data = []
            
            # If no drought collections were found, try some specific collections by ID
            if not filtered_collections['drought']:
                logger.warning("No drought collections detected automatically. Trying known collection IDs...")
                for collection_id in KNOWN_DROUGHT_COLLECTIONS:
                    try:
                        collection = self.client.get_collection(collection_id)
                        if collection:
                            filtered_collections['drought'].append(collection)
                            logger.info(f"Added known drought collection: {collection_id}")
                    except Exception as e:
                        logger.warning(f"Could not add known drought collection {collection_id}: {e}")
            
            # Process EM-DAT collections
            logger.info(f"Processing {len(filtered_collections['emdat'])} EM-DAT collections...")
            for collection in filtered_collections['emdat']:
                collection_id = collection.id
                output_file, item_count = self.extract_data_for_collection(
                    collection_id=collection_id,
                    hazard_type='emdat',
                    date_range=date_range,
                    output_dir=output_dir
                )
                
                if output_file:
                    summary_data.append({
                        'hazard_type': 'emdat',
                        'collection_id': collection_id,
                        'start_date': date_range[0],
                        'end_date': date_range[1],
                        'item_count': item_count,
                        'output_file': os.path.basename(output_file)
                    })
            
            # Process flood collections
            logger.info(f"Processing {len(filtered_collections['flood'])} flood collections...")
            for collection in filtered_collections['flood']:
                collection_id = collection.id
                output_file, item_count = self.extract_data_for_collection(
                    collection_id=collection_id,
                    hazard_type='flood',
                    date_range=date_range,
                    output_dir=output_dir
                )
                
                if output_file:
                    summary_data.append({
                        'hazard_type': 'flood',
                        'collection_id': collection_id,
                        'start_date': date_range[0],
                        'end_date': date_range[1],
                        'item_count': item_count,
                        'output_file': os.path.basename(output_file)
                    })
            
            # Process drought collections
            logger.info(f"Processing {len(filtered_collections['drought'])} drought collections...")
            for collection in filtered_collections['drought']:
                collection_id = collection.id
                output_file, item_count = self.extract_data_for_collection(
                    collection_id=collection_id,
                    hazard_type='drought',
                    date_range=date_range,
                    output_dir=output_dir
                )
                
                if output_file:
                    summary_data.append({
                        'hazard_type': 'drought',
                        'collection_id': collection_id,
                        'start_date': date_range[0],
                        'end_date': date_range[1],
                        'item_count': item_count,
                        'output_file': os.path.basename(output_file)
                    })
            
            # Save summary to CSV
            summary_df = pd.DataFrame(summary_data)
            if not summary_df.empty:
                summary_df.to_csv(summary_file, index=False)
                logger.info(f"Extraction complete. Summary saved to {summary_file}")
            else:
                logger.warning("No data extracted for any collections.")
            
            # Create a combined CSV with all hazard data
            self.combine_hazard_data(output_dir)
            
            return summary_file
            
        except Exception as e:
            logger.error(f"Failed to extract hazard data: {e}")
            raise
    
    def combine_hazard_data(self, data_dir: str):
        """Combine all hazard CSV files into one master file."""
        try:
            combined_file = os.path.join(data_dir, "all_hazard_data.csv")
            metadata_file = os.path.join(data_dir, "metadata.txt")
            
            # Create metadata file with extraction information
            with open(metadata_file, "w") as f:
                f.write("ICPAC Flood and Drought Data Extraction\n")
                f.write("======================================\n\n")
                f.write(f"Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Date Range: 2022-01-01 to {datetime.now().strftime('%Y-%m-%d')}\n")
                f.write(f"Output Directory: {os.path.abspath(data_dir)}\n\n")
                f.write("This data contains flood and drought events extracted from the Montandon STAC API.\n")
                f.write("Data includes EM-DAT disaster records where available.\n")
            
            # Find all CSV files in the directory (excluding summary file)
            csv_files = [f for f in os.listdir(data_dir) 
                        if f.endswith('.csv') and f != "extraction_summary.csv"]
            
            if not csv_files:
                logger.warning("No CSV files found to combine.")
                return None
            
            # Combine all CSV files
            dfs = []
            for file in csv_files:
                try:
                    file_path = os.path.join(data_dir, file)
                    df = pd.read_csv(file_path)
                    if not df.empty:
                        dfs.append(df)
                except Exception as e:
                    logger.error(f"Error reading {file}: {e}")
            
            if not dfs:
                logger.warning("No valid data found in CSV files.")
                return None
            
            # Combine all dataframes
            combined_df = pd.concat(dfs, ignore_index=True)
            
            # Save combined data
            combined_df.to_csv(combined_file, index=False)
            logger.info(f"Combined {len(dfs)} CSV files into {combined_file}")
            
            # Create statistics summary
            self._create_statistics_summary(combined_df, data_dir)
            
            return combined_file
            
        except Exception as e:
            logger.error(f"Failed to combine hazard data: {e}")
            return None
            
    def _create_statistics_summary(self, df, data_dir):
        """Create a statistics summary of the extracted data."""
        try:
            stats_file = os.path.join(data_dir, "statistics_summary.csv")
            
            # Calculate statistics by country and hazard type
            stats = []
            
            # Group by country and hazard_type
            grouped = df.groupby(['country', 'hazard_type'])
            
            for (country, hazard_type), group in grouped:
                stats.append({
                    'country': country,
                    'hazard_type': hazard_type,
                    'count': len(group),
                    'earliest_date': group['date'].min() if 'date' in group.columns else 'N/A',
                    'latest_date': group['date'].max() if 'date' in group.columns else 'N/A',
                    'collections': len(group['collection_id'].unique() if 'collection_id' in group.columns else [])
                })
            
            # Save statistics
            stats_df = pd.DataFrame(stats)
            stats_df.to_csv(stats_file, index=False)
            logger.info(f"Created statistics summary at {stats_file}")
            
        except Exception as e:
            logger.error(f"Failed to create statistics summary: {e}")


def get_date_range(custom_start: str = None, days: int = 90) -> Tuple[str, str]:
    """
    Get a date range from now to either a custom start date or N days in the past.
    
    Args:
        custom_start: Custom start date in 'YYYY-MM-DD' format
        days: Number of days in the past (used only if custom_start is None)
    """
    end_date = datetime.now().strftime('%Y-%m-%d')
    
    if custom_start:
        start_date = custom_start
    else:
        start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        
    return start_date, end_date


def main():
    """Main function for extracting flood and drought data."""
    try:
        # Initialize components
        client = STACClient()
        processor = CSVDataProcessor(client)
        extractor = HazardDataExtractor(client, processor)

        # Print available flood and drought collections
        print("Listing available flood, drought, and EM-DAT collections...")
        client.print_hazard_collections()

        # Set date range from January 1, 2022 to current date
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = '2022-01-01'
        date_range = (start_date, end_date)
        print(f"\nExtracting data from {date_range[0]} to {date_range[1]}")

        # Extract all flood and drought data
        output_dir = "hazard_data"
        summary_file = extractor.extract_all_hazard_data(date_range, output_dir)
        
        if summary_file:
            print(f"\nExtraction complete. Results saved to {output_dir} directory.")
            print(f"Summary file: {summary_file}")
            
            # Check if we found all types of data
            combined_file = os.path.join(output_dir, "all_hazard_data.csv")
            if os.path.exists(combined_file):
                print(f"Combined data file: {combined_file}")
                
                try:
                    df = pd.read_csv(combined_file)
                    drought_count = len(df[df['hazard_type'] == 'drought'])
                    flood_count = len(df[df['hazard_type'] == 'flood'])
                    emdat_count = len(df[df['hazard_type'] == 'emdat'])
                    
                    print(f"\nData summary:")
                    print(f"  - Flood events: {flood_count}")
                    print(f"  - Drought events: {drought_count}")
                    print(f"  - EM-DAT records: {emdat_count}")
                    
                    if drought_count == 0:
                        print("\nNOTE: No drought events were found.")
                    if flood_count == 0:
                        print("\nNOTE: No flood events were found.")
                    if emdat_count == 0:
                        print("\nNOTE: No EM-DAT records were found.")
                        
                except Exception as e:
                    print(f"Error analyzing results: {e}")
        else:
            print("Extraction failed or no data was found.")

    except Exception as e:
        logger.error(f"An error occurred in the main function: {e}")
        print(f"Error: {e}")
        
    print("\nDone.")


if __name__ == "__main__":
    main()

2025-04-14 09:31:38,916 - INFO - Connected to STAC API at https://montandon-eoapi-stage.ifrc.org/stac


Listing available flood, drought, and EM-DAT collections...


2025-04-14 09:31:39,503 - INFO - Found 29 collections
2025-04-14 09:31:39,505 - INFO - Found EM-DAT collection: emdat-events
2025-04-14 09:31:39,505 - INFO - Found EM-DAT collection: emdat-hazards
2025-04-14 09:31:39,506 - INFO - Found EM-DAT collection: emdat-impacts
2025-04-14 09:31:39,507 - INFO - Found 7 flood-related collections
2025-04-14 09:31:39,507 - INFO - Found 1 drought-related collections
2025-04-14 09:31:39,508 - INFO - Found 3 EM-DAT collections
2025-04-14 09:31:39,509 - INFO - Flood collections: ['gdacs-events', 'gdacs-hazards', 'gfd-events', 'gfd-hazards', 'gfd-impacts', 'pdc-events', 'pdc-hazards']
2025-04-14 09:31:39,510 - INFO - Drought collections: ['pdc-impacts']
2025-04-14 09:31:39,510 - INFO - EM-DAT collections: ['emdat-events', 'emdat-hazards', 'emdat-impacts']



Flood-Related Collections:
ID: gdacs-events
Title: GDACS Source Events
Description: Events from the Global Disaster Alert and Coordination System (GDACS). GDACS is a cooperation framew...
--------------------------------------------------
ID: gdacs-hazards
Title: GDACS Hazards
Description: Hazard records from the Global Disaster Alert and Coordination System (GDACS). GDACS is a cooperatio...
--------------------------------------------------
ID: gfd-events
Title: GFD Source Events
Description: Events from the Global Flood Database (GFD). GFD combines several years of flood data to create a co...
--------------------------------------------------
ID: gfd-hazards
Title: GFD Source Hazards
Description: Hazard records from the Global Flood Database (GFD). GFD combines several years of flood data to cre...
--------------------------------------------------
ID: gfd-impacts
Title: GFD Impacts
Description: Impact records from the Global Flood Database (GFD). GFD combines several years of floo

2025-04-14 09:31:39,843 - INFO - Found 29 collections
2025-04-14 09:31:39,844 - INFO - Found EM-DAT collection: emdat-events
2025-04-14 09:31:39,844 - INFO - Found EM-DAT collection: emdat-hazards
2025-04-14 09:31:39,845 - INFO - Found EM-DAT collection: emdat-impacts
2025-04-14 09:31:39,846 - INFO - Found 7 flood-related collections
2025-04-14 09:31:39,847 - INFO - Found 1 drought-related collections
2025-04-14 09:31:39,848 - INFO - Found 3 EM-DAT collections
2025-04-14 09:31:39,848 - INFO - Flood collections: ['gdacs-events', 'gdacs-hazards', 'gfd-events', 'gfd-hazards', 'gfd-impacts', 'pdc-events', 'pdc-hazards']
2025-04-14 09:31:39,849 - INFO - Drought collections: ['pdc-impacts']
2025-04-14 09:31:39,850 - INFO - EM-DAT collections: ['emdat-events', 'emdat-hazards', 'emdat-impacts']
2025-04-14 09:31:39,851 - INFO - Processing 3 EM-DAT collections...
2025-04-14 09:31:39,851 - INFO - Extracting emdat data from collection 'emdat-events' (2022-01-01 to 2025-04-14)
Searching emdat-event


Extraction complete. Results saved to hazard_data directory.
Summary file: hazard_data/extraction_summary.csv
Combined data file: hazard_data/all_hazard_data.csv

Data summary:
  - Flood events: 180
  - Drought events: 20
  - EM-DAT records: 16

Done.
